# Publication copy

Outputs and machine-specific paths were removed. Run `python scripts/prepare_local_artifacts.py` first. GPU experiments additionally require datasets, authorized pretrained models and the large artifacts listed in `docs/LARGE_ARTIFACTS.md`. Do not overwrite the frozen reporting inputs.


In [ ]:
from pathlib import Path
import os, sys
_start = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).expanduser().resolve()
_PUBLICATION_ROOT = next((p for p in (_start, *_start.parents)
                         if (p / "Methods").is_dir() and (p / "README.md").is_file()), None)
if _PUBLICATION_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT to the cloned cancer_image_pathology folder")
os.chdir(_PUBLICATION_ROOT)
sys.path.insert(0, str(_PUBLICATION_ROOT))
os.environ["PROJECT_ROOT"] = str(_PUBLICATION_ROOT)
print("Project:", _PUBLICATION_ROOT)


# Feedback2 probability deletion sensitivity: deterministic ties (v5)

Reuses saved checkpoints, true-class attribution rankings, cohorts, seeds 11/89/181,
normalized-zero replacement, and the common 14 x 14 grid. No model training, no new
attribution method, no attribution-map regeneration. This sensitivity complements
occlusion Spearman and deletion success; probability changes still depend on calibration.

Equal scores are ordered by ascending patch index for every model. This is a
declared tie-handling refinement, not an exact replay of historical tied steps.
Original patch permutations were not archived. Baseline, random deletion, and
every tie-free deletion step must still match. Both new logit and probability
curves are saved so comparisons use identical interventions.

Run the proof cell first. Then enable RUN_FULL and run the full cell. Results resume
by checkpoint partition and are isolated from all previous artifacts. Upload the new
Methods/Feedback2 folder plus this notebook; keep the existing Methods dependencies,
model checkpoints, datasets, and Kather/CRC artifacts online. Download
artifacts/feedback2_revision/probability_deletion when complete.


In [ ]:
from pathlib import Path
import os, sys
import torch
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '.'))
assert (PROJECT_ROOT / 'Methods/Feedback2').is_dir(), PROJECT_ROOT
sys.path.insert(0, str(PROJECT_ROOT))
from Methods.Feedback2.probability_deletion import inputs, select_metrics, run_dataset, configure_replay_backend, archive_curve_path, OUTPUT_SUBDIR
DEVICE = torch.device('cuda:0')
assert torch.cuda.is_available(), 'Run this notebook in the online GPU environment.'
DATASETS = ['kather', 'crc']
MODELS = ['ResNet18', 'DINOv2', 'UNI']
UNI_ASSETS = os.environ.get('UNI_ASSETS_DIR') or None
BATCH_SIZE = 8
RUN_FULL = False
print(configure_replay_backend())
print(torch.cuda.get_device_name(DEVICE))


In [ ]:
# Saved inputs only; no model is loaded in this preflight.
import unittest
from Methods.Feedback2.test_probability_deletion import ProbabilityDeletionTests
result = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(ProbabilityDeletionTests))
assert result.wasSuccessful(), 'Probability replay unit tests failed.'
for dataset in DATASETS:
    metric_path, map_path = inputs(PROJECT_ROOT, dataset)
    assert map_path.is_file(), map_path
    assert archive_curve_path(PROJECT_ROOT, dataset).is_file(), archive_curve_path(PROJECT_ROOT, dataset)
    frame = select_metrics(metric_path)
    display(frame.groupby(['cohort_name', 'analysis_family', 'model']).agg(
        images=('cohort_id', 'nunique'), observations=('seed', 'size')))


In [ ]:
# Check the previously failing target before the six proof runs.
from Methods.Feedback2.probability_deletion import make_adapter, image_path, validate_baseline, KEY
frame = select_metrics(inputs(PROJECT_ROOT, 'kather')[0], 'ResNet18')
row = frame.loc[frame.cohort_id.eq('dc1d4e475494529d') & frame.seed.eq(11)].iloc[0]
adapter = make_adapter(PROJECT_ROOT, 'kather', 'ResNet18', DEVICE)
try:
    adapter.load_checkpoint(int(row.fold), int(row.seed))
    image = adapter.load_image(str(image_path(PROJECT_ROOT, 'kather', row)))
    with torch.inference_mode():
        baseline = adapter.model(image).float().cpu()[0]
    validate_baseline(baseline, row, tuple(row[k] for k in KEY))
    print('Previously failing target: baseline matches archived result.')
finally:
    adapter.model = None
    torch.cuda.empty_cache()

# Representative target plus all three reported regression images where eligible.
for dataset in DATASETS:
    for model_name in MODELS:
        run_dataset(PROJECT_ROOT, dataset, model_name, DEVICE, proof=True,
                    batch_size=BATCH_SIZE, uni_assets=UNI_ASSETS)


## Full targeted replay
After every proof run succeeds, set RUN_FULL=True above and execute the next cell.
Only saved true-class targets in the selected and random Kather cohorts and the
98-image CRC cohort are evaluated. Correctness eligibility is inherited unchanged.
The notebook stops on baseline, missing/conflicting archive, random-deletion, or
tie-free deletion mismatch. Differences at tied cutoffs are recorded, never called
exact replications. New outputs use a separate deterministic_ties_v5 subfolder;
v1-v4 results are retained but not pooled or reused. All models use the same rule.
Do not relax those checks to force a result. Five random orders are shared across
models by the original deterministic seed rule. Softmax is applied separately to
each repeat before averaging. Full class logits are saved at every deletion step.


In [ ]:
if RUN_FULL:
    for dataset in DATASETS:
        for model_name in MODELS:
            run_dataset(PROJECT_ROOT, dataset, model_name, DEVICE, proof=False,
                        batch_size=BATCH_SIZE, uni_assets=UNI_ASSETS)
else:
    print('Proof only. Set RUN_FULL=True after verifying all six proof runs.')


In [ ]:
import pandas as pd
if RUN_FULL:
    from Methods.Feedback2.probability_deletion import KEY
    for dataset in DATASETS:
        location = PROJECT_ROOT / OUTPUT_SUBDIR / 'full' / dataset
        files = sorted(location.glob('*/seed_*_fold_*_metrics.csv'))
        completed = pd.concat([pd.read_csv(p) for p in files], ignore_index=True)
        expected = select_metrics(inputs(PROJECT_ROOT, dataset)[0])
        identity = KEY + ['analysis_family']
        expected_keys = set(expected[identity].itertuples(index=False, name=None))
        actual_keys = set(completed[identity].itertuples(index=False, name=None))
        assert actual_keys == expected_keys, 'Incomplete output; resume missing partitions.'
        assert not completed.duplicated(identity).any()
        completed.to_csv(location / 'probability_image_seed_metrics.csv', index=False)
        display(completed.groupby(['cohort_name', 'analysis_family', 'model'])[
            'top_minus_random_probability_reduction_auc'].agg(['count', 'mean']))
        print('Verified complete:', location)
